In [32]:
# CA1 – Business Analytics: Tasks 2
# Scope: Task 2 (Retail Performance Analytics)
# Scenario: Operations Director
# Dependencies: pandas, numpy, matplotlib, seaborn, sklearn
# Bash:python -m pip install numpy pandas scikit-learn matplotlib seaborn

In [33]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the three datasets
customers = pd.read_csv("../Data/Customers_members.csv")
products = pd.read_csv("../Data/Products_overview.csv")
transactions = pd.read_csv("../Data/Transactions_history.csv")


In [34]:
# 2. Inspect dataset sizes and missing values
for name, data in [
    ("Customers", customers),
    ("Products", products),
    ("Transactions", transactions)
]:
    print(f"\n{name} dataset size:", data.shape)
    print(data.head())
    print("Missing values:")
    print(data.isnull().sum())


Customers dataset size: (1000, 5)
   CustomerID AgeGroup Region LoyaltyLevel  MembershipYears
0           1    36-50   East       Bronze                9
1           2    26-35   East       Bronze                4
2           3    18-25   East       Silver               10
3           4    36-50   East       Bronze               14
4           5    26-35  North       Bronze                7
Missing values:
CustomerID         0
AgeGroup           0
Region             0
LoyaltyLevel       0
MembershipYears    0
dtype: int64

Products dataset size: (200, 6)
   ProductID     Category     Brand   Price  Rating  PopularityScore
0          1  Electronics   TechMax  474.45    4.54                3
1          2       Gaming  SmartBuy  285.11    3.16                2
2          3        books   HomePro  370.79    3.01                1
3          4         Home   TechMax  416.38    3.38                1
4          5        Books   TechMax   62.37    3.42                1
Missing values:
ProductI

In [35]:
# Clean data by removing exact duplicate rows
customers = customers.drop_duplicates()
products = products.drop_duplicates()
transactions = transactions.drop_duplicates()

In [36]:
# 3. Combine transactions with customer and product information
data = transactions.merge(customers, on="CustomerID", how="left")
data = data.merge(products, on="ProductID", how="left")

print("\nCombined data size:", data.shape)
print("Rows with missing customer details:", data["AgeGroup"].isnull().sum())
print("Rows with missing product details:", data["Category"].isnull().sum())


Combined data size: (18782, 14)
Rows with missing customer details: 0
Rows with missing product details: 0


In [37]:
# 4. Customer analytics
print("\nTop 10 customers by total spend:")
print(data.groupby("CustomerID")["Spend"].sum().sort_values(ascending=False).head(10))

print("\nTop 10 customers by purchase frequency:")
print(data.groupby("CustomerID")["PurchaseCount"].sum().sort_values(ascending=False).head(10))


Top 10 customers by total spend:
CustomerID
395    25504.83
751    23084.09
298    22883.98
454    22750.97
57     22713.06
546    22399.85
958    22132.06
931    22050.15
177    21954.58
879    21698.92
Name: Spend, dtype: float64

Top 10 customers by purchase frequency:
CustomerID
879    81
812    81
550    81
664    80
480    80
104    79
831    79
912    79
656    79
395    79
Name: PurchaseCount, dtype: int64


In [38]:
# Customer value summary: spend and number of different products purchased
customer_value = data.groupby("CustomerID").agg(
    TotalSpend=("Spend", "sum"),
    TotalPurchases=("PurchaseCount", "sum"),
    DifferentProducts=("ProductID", "nunique")
).sort_values("TotalSpend", ascending=False)
print("\nCustomer value (top 10):")
print(customer_value.head(10))


Customer value (top 10):
            TotalSpend  TotalPurchases  DifferentProducts
CustomerID                                               
395           25504.83              79                 23
751           23084.09              72                 19
298           22883.98              76                 22
454           22750.97              76                 20
57            22713.06              71                 24
546           22399.85              79                 24
958           22132.06              76                 18
931           22050.15              76                 24
177           21954.58              74                 25
879           21698.92              81                 24


In [39]:
# 5. Product analytics
print("\nTop 10 products by revenue:")
print(data.groupby("ProductID")["Spend"].sum().sort_values(ascending=False).head(10))

print("\nRevenue by category:")
print(data.groupby("Category")["Spend"].sum().sort_values(ascending=False))

print("\nRevenue by brand:")
print(data.groupby("Brand")["Spend"].sum().sort_values(ascending=False))


Top 10 products by revenue:
ProductID
102    1718993.64
7       476891.01
41      432435.78
134     405060.82
8       234323.88
184     189363.83
45      185750.73
65      184635.64
81      178441.35
137     169315.65
Name: Spend, dtype: float64

Revenue by category:
Category
Books          2561209.10
Home           2289506.39
Electronics    1680454.38
Fitness        1306839.94
Gaming         1295944.49
Kitchen        1273372.42
kitchen         492021.64
fitness         238465.24
books           178206.60
electronics      85223.50
gaming           85026.42
home              2474.00
Name: Spend, dtype: float64

Revenue by brand:
Brand
FitLife       3283248.33
SmartBuy      2576635.12
HomePro       1862715.87
TechMax       1600803.26
NextGen       1272086.37
PrimeGoods     893255.17
Name: Spend, dtype: float64


In [40]:
# 6. Three GroupBy analyses (customer, category, brand) are above.
# Two pivot tables:
category_pivot = pd.pivot_table(
    data, values="Spend", index="Category", columns="LoyaltyLevel",
    aggfunc="sum", fill_value=0
)
print("\nPivot table: spend by category and loyalty level")
print(category_pivot)

brand_pivot = pd.pivot_table(
    data, values="Spend", index="Brand", columns="Region",
    aggfunc="sum", fill_value=0
)
print("\nPivot table: spend by brand and customer region")
print(brand_pivot)


Pivot table: spend by category and loyalty level
LoyaltyLevel      Bronze       Gold     Silver
Category                                      
Books         1278929.68  540110.60  742168.82
Electronics    858990.57  337726.53  483737.28
Fitness        619694.58  303400.71  383744.65
Gaming         670338.71  262552.37  363053.41
Home          1159177.10  481425.76  648903.53
Kitchen        640121.65  268489.64  364761.13
books          100218.84   34065.01   43922.75
electronics     45238.87   22904.39   17080.24
fitness        133574.85   44140.89   60749.50
gaming          41183.45   15615.72   28227.25
home             1434.92     519.54     519.54
kitchen        249725.34   99332.42  142963.88

Pivot table: spend by brand and customer region
Region           East      North      South       West
Brand                                                 
FitLife     812720.03  760510.95  852494.47  857522.88
HomePro     457931.55  460958.16  476674.38  467151.78
NextGen     335961.26  

In [41]:
# 7. Build the customer-product interaction matrix.

# Spend is used as the strength of each customer-product interaction.
interaction = data.pivot_table(
    index="CustomerID", columns="ProductID", values="Spend",
    aggfunc="sum", fill_value=0
)

# Method 1: Popularity-based recommendations.
# Recommend the products with the highest total spend, excluding products
# the customer has already purchased.
product_popularity = interaction.sum(axis=0).sort_values(ascending=False)

def popularity_recommend(customer_id, n=5):
    already_bought = set(interaction.loc[customer_id][interaction.loc[customer_id] > 0].index)
    choices = [p for p in product_popularity.index if p not in already_bought]
    return choices[:n]

# Method 2: Item-item collaborative filtering.
# Find products bought/spent on by similar customers, using cosine similarity
# between product columns. Score unseen products based on the customer's
# existing spend on similar products.
item_similarity = pd.DataFrame(
    cosine_similarity(interaction.T),
    index=interaction.columns,
    columns=interaction.columns
)

def collaborative_recommend(customer_id, n=5):
    customer_spend = interaction.loc[customer_id]
    scores = item_similarity.dot(customer_spend)
    scores[customer_spend > 0] = -1  # Exclude products already purchased
    return scores.sort_values(ascending=False).head(n).index.tolist()

In [42]:
# 8. Show recommendations for five example customers
example_customers = interaction.index[:5]

for customer_id in example_customers:
    print(f"\nCustomer {customer_id}")
    print("Popularity recommendations:", popularity_recommend(customer_id))
    print("Collaborative recommendations:", collaborative_recommend(customer_id))


Customer 1
Popularity recommendations: [102, 7, 41, 134, 8]
Collaborative recommendations: [7, 102, 41, 13, 8]

Customer 2
Popularity recommendations: [102, 41, 134, 8, 184]
Collaborative recommendations: [102, 13, 134, 41, 184]

Customer 3
Popularity recommendations: [102, 7, 41, 134, 8]
Collaborative recommendations: [102, 7, 13, 134, 195]

Customer 4
Popularity recommendations: [7, 41, 134, 8, 184]
Collaborative recommendations: [7, 13, 184, 30, 134]

Customer 5
Popularity recommendations: [102, 7, 41, 134, 8]
Collaborative recommendations: [102, 7, 41, 13, 119]


In [43]:
# 9. Simple offline evaluation with a held-out interaction.
# For customers with at least two purchased products, hide one product,
# build recommendations from the remaining history, and check if the hidden
# product appears in the top 10. This is a simple test, not a live-sales test.
def evaluate_recommenders(k=10):
    popularity_hits = 0
    collaborative_hits = 0
    tested = 0

    for customer_id in interaction.index:
        purchased = interaction.loc[customer_id]
        purchased_products = purchased[purchased > 0].index.tolist()

        if len(purchased_products) < 2:
            continue

        # Use the final product in the list as the test item.
        test_product = purchased_products[-1]
        train_spend = purchased.copy()
        train_spend[test_product] = 0

        # Popularity method: exclude products still present in training history.
        already_bought = set(train_spend[train_spend > 0].index)
        pop_choices = [p for p in product_popularity.index if p not in already_bought]
        pop_top = pop_choices[:k]

        # Collaborative method: score items using the reduced history.
        collab_scores = item_similarity.dot(train_spend)
        collab_scores[train_spend > 0] = -1
        collab_top = collab_scores.sort_values(ascending=False).head(k).index.tolist()

        popularity_hits += int(test_product in pop_top)
        collaborative_hits += int(test_product in collab_top)
        tested += 1

    if tested:
        print("\nOffline evaluation (Hit Rate@10)")
        print("Customers tested:", tested)
        print("Popularity Hit Rate@10:", round(popularity_hits / tested, 4))
        print("Collaborative Hit Rate@10:", round(collaborative_hits / tested, 4))
    else:
        print("Not enough customer histories to evaluate.")

evaluate_recommenders()


Offline evaluation (Hit Rate@10)
Customers tested: 971
Popularity Hit Rate@10: 0.1277
Collaborative Hit Rate@10: 0.4181


In [44]:
# Important: this is a simple educational recommender and offline evaluation.
# It does not prove recommendations will increase sales in real life.